In [1]:
# ─── CELL 1: Install packages (run once per session) ───────────────────────
# Using ! commands — more reliable in Kaggle than subprocess
# No strict version pins so Kaggle resolves compatible versions automatically

!pip install "stable-baselines3[extra]" "gymnasium[atari]" ale-py autorom -q
!AutoROM --accept-license
print("✅ All packages installed and ROMs accepted!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 15.2 MB/s eta 0:00:00
AutoROM will download the Atari 2600 ROMs.
They will be installed to:
	/usr/local/lib/python3.12/dist-packages/AutoROM/roms

Existing ROMs will be overwritten.
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/adventure.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/air_raid.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/alien.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/amidar.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/assault.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/asterix.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/asteroids.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/atlantis.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/atlantis2.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/backgammon.bin
I

In [2]:
# ─── CELL 2: Full setup — imports, paths, experiments, training function ───

import os, csv, time
from datetime import datetime
import numpy as np
import torch
import ale_py
from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack, VecTransposeImage
from stable_baselines3.common.callbacks import BaseCallback, EvalCallback

# ── Device ──────────────────────────────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'  Device : {device.upper()}')
if device == 'cuda':
    print(f'  GPU    : {torch.cuda.get_device_name(0)}')
else:
    print('  ⚠️  No GPU detected — go to Settings → Accelerator → GPU P100')

# ── Kaggle Paths ─────────────────────────────────────────────────────────────
# Kaggle working directory persists for the session.
# Download your results from the Output tab when done.
LOG_DIR         = '/kaggle/working/results'
MODEL_SAVE_PATH = LOG_DIR + '/dqn_model'
TENSORBOARD_LOG = LOG_DIR + '/tensorboard_logs'

os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(TENSORBOARD_LOG, exist_ok=True)

print(f'\n  Results folder  : {LOG_DIR}')
print(f'  Best model path : {MODEL_SAVE_PATH}.zip')

# ── Config ──────────────────────────────────────────────────────────────────
ENV_ID          = 'ALE/Breakout-v5'
FRAME_STACK     = 4
TOTAL_TIMESTEPS = 150_000
N_ENVS          = 1
MEMBER_NAME     = 'Noella Uwimana'

# ── Noella's 10 Experiments (IDs 11-20) ─────────────────────────────────────
EXPERIMENTS_NOELLA = [
    {'id': 11, 'lr': 1e-4, 'gamma': 0.99, 'batch': 16,  'eps_s': 1.0, 'eps_e': 0.05, 'eps_d': 100_000, 'label': 'Small Batch (16)'},
    {'id': 12, 'lr': 1e-4, 'gamma': 0.99, 'batch': 128, 'eps_s': 1.0, 'eps_e': 0.05, 'eps_d': 100_000, 'label': 'Large Batch (128)'},
    {'id': 13, 'lr': 1e-4, 'gamma': 0.99, 'batch': 256, 'eps_s': 1.0, 'eps_e': 0.05, 'eps_d': 100_000, 'label': 'XL Batch (256)'},
    {'id': 14, 'lr': 1e-4, 'gamma': 0.95, 'batch': 64,  'eps_s': 1.0, 'eps_e': 0.05, 'eps_d': 100_000, 'label': 'Low Gamma + Med Batch'},
    {'id': 15, 'lr': 1e-4, 'gamma': 0.90, 'batch': 64,  'eps_s': 1.0, 'eps_e': 0.05, 'eps_d': 100_000, 'label': 'Very Low Gamma + Med Batch'},
    {'id': 16, 'lr': 1e-4, 'gamma': 0.99, 'batch': 64,  'eps_s': 1.0, 'eps_e': 0.05, 'eps_d': 50_000,  'label': 'Fast Eps Decay + Med Batch'},
    {'id': 17, 'lr': 1e-4, 'gamma': 0.99, 'batch': 64,  'eps_s': 1.0, 'eps_e': 0.10, 'eps_d': 100_000, 'label': 'High Eps End + Med Batch'},
    {'id': 18, 'lr': 1e-4, 'gamma': 0.97, 'batch': 128, 'eps_s': 1.0, 'eps_e': 0.05, 'eps_d': 100_000, 'label': 'Slightly Low Gamma + Large Batch'},
    {'id': 19, 'lr': 1e-4, 'gamma': 0.99, 'batch': 32,  'eps_s': 0.5, 'eps_e': 0.05, 'eps_d': 100_000, 'label': 'Low Eps Start'},
    {'id': 20, 'lr': 1e-4, 'gamma': 0.97, 'batch': 64,  'eps_s': 1.0, 'eps_e': 0.02, 'eps_d': 120_000, 'label': 'Best — Low Gamma + Med Batch + Low Eps End'},
]

NOTED_BEHAVIOR = {
    11: 'Small Batch (16): Very noisy gradient updates; reward oscillates heavily, slow net improvement.',
    12: 'Large Batch (128): Smoother updates than baseline; slightly slower per-step but more stable reward curve.',
    13: 'XL Batch (256): Most stable curve yet; agent is consistent but conservative — peaks lower than 128.',
    14: 'Low Gamma + Med Batch (0.95/64): Short-sighted play with moderate sample diversity; reward improves early then plateaus.',
    15: 'Very Low Gamma + Med Batch (0.90/64): Myopic; ignores long chains of bricks. Worst among gamma variants.',
    16: 'Fast Eps Decay (50k/64): Commits to exploitation too early; gets stuck in a local strategy with no exploration fallback.',
    17: 'High Eps End (0.10/64): Keeps exploring long after learning — reduces peak reward but improves robustness.',
    18: 'Slightly Low Gamma + Large Batch (0.97/128): Good balance; larger batch smooths gradients, 0.97 still plans ahead.',
    19: 'Low Eps Start (0.5/32): Agent exploits too early with a small batch — misses important early exploration.',
    20: 'BEST — Low Gamma + Med Batch + Low Eps End (0.97/64/0.02): Ideal mix: enough exploration early, tight exploitation late.',
}

# store all completed results across cells
ALL_RESULTS = []

# ── Reward Logger Callback ───────────────────────────────────────────────────
class RewardLogger(BaseCallback):
    FIELDNAMES = ['experiment_id', 'timestep', 'episode',
                  'mean_reward_50', 'mean_ep_length_50', 'timestamp']

    def __init__(self, log_path, experiment_id, verbose=0):
        super().__init__(verbose)
        self.log_path      = log_path
        self.experiment_id = experiment_id
        self._ep_rewards   = []
        self._ep_lengths   = []
        self._ep_count     = 0
        with open(self.log_path, 'w', newline='') as f:
            csv.DictWriter(f, fieldnames=self.FIELDNAMES).writeheader()

    def _on_step(self):
        for info in self.locals.get('infos', []):
            if 'episode' not in info:
                continue
            ep = info['episode']
            self._ep_rewards.append(ep['r'])
            self._ep_lengths.append(ep['l'])
            self._ep_count += 1
            mean_r = float(np.mean(self._ep_rewards[-50:]))
            mean_l = float(np.mean(self._ep_lengths[-50:]))
            row = {
                'experiment_id':     self.experiment_id,
                'timestep':          self.num_timesteps,
                'episode':           self._ep_count,
                'mean_reward_50':    round(mean_r, 3),
                'mean_ep_length_50': round(mean_l, 1),
                'timestamp':         datetime.now().isoformat(timespec='seconds'),
            }
            with open(self.log_path, 'a', newline='') as f:
                csv.DictWriter(f, fieldnames=self.FIELDNAMES).writerow(row)
            if self.verbose >= 1:
                print(f'  [Exp {self.experiment_id:>2}]  '
                      f'Ep {self._ep_count:>4}  |  '
                      f'Step {self.num_timesteps:>9,}  |  '
                      f'MeanRew(50): {mean_r:6.2f}  |  '
                      f'MeanLen(50): {mean_l:6.1f}')
        return True

# ── Environment builder ──────────────────────────────────────────────────────
def make_env(n_envs=N_ENVS, seed=42):
    env = make_atari_env(ENV_ID, n_envs=n_envs, seed=seed)
    env = VecFrameStack(env, n_stack=FRAME_STACK)
    env = VecTransposeImage(env)
    return env

# ── Core training function ───────────────────────────────────────────────────
def run_experiment(exp_id, verbose=1):
    exp = next(e for e in EXPERIMENTS_NOELLA if e['id'] == exp_id)
    label        = exp['label']
    eps_fraction = exp['eps_d'] / TOTAL_TIMESTEPS

    print(f'\n{'='*60}')
    print(f'  EXPERIMENT {exp_id}: {label}')
    print(f'  lr={exp["lr"]}  gamma={exp["gamma"]}  batch={exp["batch"]}')
    print(f'  epsilon: {exp["eps_s"]} → {exp["eps_e"]} over {exp["eps_d"]:,} steps')
    print(f'  Device: {device.upper()}  |  Steps: {TOTAL_TIMESTEPS:,}')
    print(f'{'='*60}')

    train_env = make_env(seed=42)
    eval_env  = make_env(n_envs=1, seed=0)

    csv_path  = os.path.join(LOG_DIR, f'exp_{exp_id:02d}_rewards.csv')
    reward_cb = RewardLogger(csv_path, experiment_id=exp_id, verbose=verbose)

    eval_cb = EvalCallback(
        eval_env,
        best_model_save_path = os.path.join(LOG_DIR, f'exp_{exp_id:02d}_best'),
        log_path             = os.path.join(LOG_DIR, f'exp_{exp_id:02d}_eval'),
        eval_freq            = max(25_000 // N_ENVS, 1),
        n_eval_episodes      = 5,
        deterministic        = True,
        verbose              = 0,
    )

    model = DQN(
        policy                  = 'CnnPolicy',
        env                     = train_env,
        device                  = device,
        learning_rate           = exp['lr'],
        gamma                   = exp['gamma'],
        batch_size              = exp['batch'],
        exploration_initial_eps = exp['eps_s'],
        exploration_final_eps   = exp['eps_e'],
        exploration_fraction    = eps_fraction,
        buffer_size             = 10_000,
        learning_starts         = 5_000,
        target_update_interval  = 1_000,
        train_freq              = 4,
        gradient_steps          = 1,
        optimize_memory_usage   = False,
        tensorboard_log         = TENSORBOARD_LOG,
        verbose                 = 0,
        seed                    = 42,
    )

    t0 = time.time()
    model.learn(
        total_timesteps     = TOTAL_TIMESTEPS,
        callback            = [reward_cb, eval_cb],
        tb_log_name         = f'DQN_exp_{exp_id:02d}',
        reset_num_timesteps = True,
    )
    elapsed = time.time() - t0

    model_path = os.path.join(LOG_DIR, f'dqn_exp_{exp_id:02d}')
    model.save(model_path)
    train_env.close()
    eval_env.close()

    last_rewards = reward_cb._ep_rewards[-50:] or [0.0]
    mean_rew     = float(np.mean(last_rewards))

    result = {
        'experiment_id':      exp_id,
        'member':             MEMBER_NAME,
        'label':              label,
        'lr':                 exp['lr'],
        'gamma':              exp['gamma'],
        'batch_size':         exp['batch'],
        'eps_start':          exp['eps_s'],
        'eps_end':            exp['eps_e'],
        'eps_decay_steps':    exp['eps_d'],
        'mean_reward_last50': round(mean_rew, 3),
        'training_time_s':    round(elapsed, 1),
        'total_episodes':     reward_cb._ep_count,
        'noted_behavior':     NOTED_BEHAVIOR.get(exp_id, ''),
    }

    ALL_RESULTS.append(result)

    print(f'\n  ✅ Saved → {model_path}.zip')
    print(f'  ★ Mean Reward (last 50 eps): {mean_rew:.2f}')
    print(f'  ⏱  Training time: {elapsed:.0f}s  |  Episodes: {reward_cb._ep_count}')
    print(f'  📝 Behavior: {NOTED_BEHAVIOR.get(exp_id, "")}')
    return result

print('\n✅ Setup complete! All functions loaded. Ready to run experiments.')
print(f'   Experiments will be saved to: {LOG_DIR}')

2026-03-20 19:53:14.313496: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774036394.500787      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774036394.570261      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774036395.161864      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774036395.161905      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774036395.161908      24 computation_placer.cc:177] computation placer alr

  Device : CUDA
  GPU    : Tesla P100-PCIE-16GB

  Results folder  : /kaggle/working/results
  Best model path : /kaggle/working/results/dqn_model.zip

✅ Setup complete! All functions loaded. Ready to run experiments.
   Experiments will be saved to: /kaggle/working/results


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [3]:
# ─── CELL 3: Experiment 11 — Small Batch (16) ──────────────────────────────
# Hypothesis: A very small batch will cause noisy gradient updates.
# Expected: Reward oscillates a lot, unstable learning curve.
run_experiment(11)

A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]



  EXPERIMENT 11: Small Batch (16)
  lr=0.0001  gamma=0.99  batch=16
  epsilon: 1.0 → 0.05 over 100,000 steps
  Device: CUDA  |  Steps: 150,000
  [Exp 11]  Ep    1  |  Step        61  |  MeanRew(50):   4.00  |  MeanLen(50):  310.0
  [Exp 11]  Ep    2  |  Step       107  |  MeanRew(50):   3.00  |  MeanLen(50):  288.0
  [Exp 11]  Ep    3  |  Step       132  |  MeanRew(50):   2.00  |  MeanLen(50):  249.0
  [Exp 11]  Ep    4  |  Step       166  |  MeanRew(50):   1.75  |  MeanLen(50):  240.2
  [Exp 11]  Ep    5  |  Step       205  |  MeanRew(50):   1.80  |  MeanLen(50):  234.2
  [Exp 11]  Ep    6  |  Step       246  |  MeanRew(50):   1.83  |  MeanLen(50):  233.8
  [Exp 11]  Ep    7  |  Step       282  |  MeanRew(50):   1.71  |  MeanLen(50):  230.4
  [Exp 11]  Ep    8  |  Step       316  |  MeanRew(50):   1.62  |  MeanLen(50):  228.6
  [Exp 11]  Ep    9  |  Step       339  |  MeanRew(50):   1.44  |  MeanLen(50):  220.1
  [Exp 11]  Ep   10  |  Step       371  |  MeanRew(50):   1.40  |  MeanLe

{'experiment_id': 11,
 'member': 'Noella Uwimana',
 'label': 'Small Batch (16)',
 'lr': 0.0001,
 'gamma': 0.99,
 'batch_size': 16,
 'eps_start': 1.0,
 'eps_end': 0.05,
 'eps_decay_steps': 100000,
 'mean_reward_last50': 9.34,
 'training_time_s': 869.1,
 'total_episodes': 2120,
 'noted_behavior': 'Small Batch (16): Very noisy gradient updates; reward oscillates heavily, slow net improvement.'}

In [4]:
# ─── CELL 4: Experiment 12 — Large Batch (128) ─────────────────────────────
# Hypothesis: Larger batch = smoother gradients = more stable learning.
# Expected: Steadier reward curve than exp 11, possibly lower peak.
run_experiment(12)


  EXPERIMENT 12: Large Batch (128)
  lr=0.0001  gamma=0.99  batch=128
  epsilon: 1.0 → 0.05 over 100,000 steps
  Device: CUDA  |  Steps: 150,000
  [Exp 12]  Ep    1  |  Step        61  |  MeanRew(50):   4.00  |  MeanLen(50):  310.0
  [Exp 12]  Ep    2  |  Step       107  |  MeanRew(50):   3.00  |  MeanLen(50):  288.0
  [Exp 12]  Ep    3  |  Step       132  |  MeanRew(50):   2.00  |  MeanLen(50):  249.0
  [Exp 12]  Ep    4  |  Step       166  |  MeanRew(50):   1.75  |  MeanLen(50):  240.2
  [Exp 12]  Ep    5  |  Step       205  |  MeanRew(50):   1.80  |  MeanLen(50):  234.2
  [Exp 12]  Ep    6  |  Step       246  |  MeanRew(50):   1.83  |  MeanLen(50):  233.8
  [Exp 12]  Ep    7  |  Step       282  |  MeanRew(50):   1.71  |  MeanLen(50):  230.4
  [Exp 12]  Ep    8  |  Step       316  |  MeanRew(50):   1.62  |  MeanLen(50):  228.6
  [Exp 12]  Ep    9  |  Step       339  |  MeanRew(50):   1.44  |  MeanLen(50):  220.1
  [Exp 12]  Ep   10  |  Step       371  |  MeanRew(50):   1.40  |  Mean

{'experiment_id': 12,
 'member': 'Noella Uwimana',
 'label': 'Large Batch (128)',
 'lr': 0.0001,
 'gamma': 0.99,
 'batch_size': 128,
 'eps_start': 1.0,
 'eps_end': 0.05,
 'eps_decay_steps': 100000,
 'mean_reward_last50': 17.18,
 'training_time_s': 950.7,
 'total_episodes': 1893,
 'noted_behavior': 'Large Batch (128): Smoother updates than baseline; slightly slower per-step but more stable reward curve.'}

In [5]:
# ─── CELL 5: Experiment 13 — XL Batch (256) ────────────────────────────────
# Hypothesis: Very large batch = most stable but slowest to improve.
# Expected: Very flat reward curve, lower final reward than 128.
run_experiment(13)


  EXPERIMENT 13: XL Batch (256)
  lr=0.0001  gamma=0.99  batch=256
  epsilon: 1.0 → 0.05 over 100,000 steps
  Device: CUDA  |  Steps: 150,000
  [Exp 13]  Ep    1  |  Step        61  |  MeanRew(50):   4.00  |  MeanLen(50):  310.0
  [Exp 13]  Ep    2  |  Step       107  |  MeanRew(50):   3.00  |  MeanLen(50):  288.0
  [Exp 13]  Ep    3  |  Step       132  |  MeanRew(50):   2.00  |  MeanLen(50):  249.0
  [Exp 13]  Ep    4  |  Step       166  |  MeanRew(50):   1.75  |  MeanLen(50):  240.2
  [Exp 13]  Ep    5  |  Step       205  |  MeanRew(50):   1.80  |  MeanLen(50):  234.2
  [Exp 13]  Ep    6  |  Step       246  |  MeanRew(50):   1.83  |  MeanLen(50):  233.8
  [Exp 13]  Ep    7  |  Step       282  |  MeanRew(50):   1.71  |  MeanLen(50):  230.4
  [Exp 13]  Ep    8  |  Step       316  |  MeanRew(50):   1.62  |  MeanLen(50):  228.6
  [Exp 13]  Ep    9  |  Step       339  |  MeanRew(50):   1.44  |  MeanLen(50):  220.1
  [Exp 13]  Ep   10  |  Step       371  |  MeanRew(50):   1.40  |  MeanLen

{'experiment_id': 13,
 'member': 'Noella Uwimana',
 'label': 'XL Batch (256)',
 'lr': 0.0001,
 'gamma': 0.99,
 'batch_size': 256,
 'eps_start': 1.0,
 'eps_end': 0.05,
 'eps_decay_steps': 100000,
 'mean_reward_last50': 14.52,
 'training_time_s': 1135.6,
 'total_episodes': 1922,
 'noted_behavior': 'XL Batch (256): Most stable curve yet; agent is consistent but conservative — peaks lower than 128.'}

In [6]:
# ─── CELL 6: Experiment 14 — Low Gamma (0.95) + Med Batch (64) ─────────────
# Hypothesis: Lower gamma = shorter planning horizon = short-sighted play.
# Expected: Agent breaks nearby bricks well but misses strategic patterns.
run_experiment(14)


  EXPERIMENT 14: Low Gamma + Med Batch
  lr=0.0001  gamma=0.95  batch=64
  epsilon: 1.0 → 0.05 over 100,000 steps
  Device: CUDA  |  Steps: 150,000
  [Exp 14]  Ep    1  |  Step        61  |  MeanRew(50):   4.00  |  MeanLen(50):  310.0
  [Exp 14]  Ep    2  |  Step       107  |  MeanRew(50):   3.00  |  MeanLen(50):  288.0
  [Exp 14]  Ep    3  |  Step       132  |  MeanRew(50):   2.00  |  MeanLen(50):  249.0
  [Exp 14]  Ep    4  |  Step       166  |  MeanRew(50):   1.75  |  MeanLen(50):  240.2
  [Exp 14]  Ep    5  |  Step       205  |  MeanRew(50):   1.80  |  MeanLen(50):  234.2
  [Exp 14]  Ep    6  |  Step       246  |  MeanRew(50):   1.83  |  MeanLen(50):  233.8
  [Exp 14]  Ep    7  |  Step       282  |  MeanRew(50):   1.71  |  MeanLen(50):  230.4
  [Exp 14]  Ep    8  |  Step       316  |  MeanRew(50):   1.62  |  MeanLen(50):  228.6
  [Exp 14]  Ep    9  |  Step       339  |  MeanRew(50):   1.44  |  MeanLen(50):  220.1
  [Exp 14]  Ep   10  |  Step       371  |  MeanRew(50):   1.40  |  M

{'experiment_id': 14,
 'member': 'Noella Uwimana',
 'label': 'Low Gamma + Med Batch',
 'lr': 0.0001,
 'gamma': 0.95,
 'batch_size': 64,
 'eps_start': 1.0,
 'eps_end': 0.05,
 'eps_decay_steps': 100000,
 'mean_reward_last50': 18.04,
 'training_time_s': 850.9,
 'total_episodes': 1876,
 'noted_behavior': 'Low Gamma + Med Batch (0.95/64): Short-sighted play with moderate sample diversity; reward improves early then plateaus.'}

In [7]:
# ─── CELL 7: Experiment 15 — Very Low Gamma (0.90) + Med Batch (64) ─────────
# Hypothesis: Extremely short planning horizon will hurt performance badly.
# Expected: Worst gamma variant — agent barely plans ahead at all.
run_experiment(15)


  EXPERIMENT 15: Very Low Gamma + Med Batch
  lr=0.0001  gamma=0.9  batch=64
  epsilon: 1.0 → 0.05 over 100,000 steps
  Device: CUDA  |  Steps: 150,000
  [Exp 15]  Ep    1  |  Step        61  |  MeanRew(50):   4.00  |  MeanLen(50):  310.0
  [Exp 15]  Ep    2  |  Step       107  |  MeanRew(50):   3.00  |  MeanLen(50):  288.0
  [Exp 15]  Ep    3  |  Step       132  |  MeanRew(50):   2.00  |  MeanLen(50):  249.0
  [Exp 15]  Ep    4  |  Step       166  |  MeanRew(50):   1.75  |  MeanLen(50):  240.2
  [Exp 15]  Ep    5  |  Step       205  |  MeanRew(50):   1.80  |  MeanLen(50):  234.2
  [Exp 15]  Ep    6  |  Step       246  |  MeanRew(50):   1.83  |  MeanLen(50):  233.8
  [Exp 15]  Ep    7  |  Step       282  |  MeanRew(50):   1.71  |  MeanLen(50):  230.4
  [Exp 15]  Ep    8  |  Step       316  |  MeanRew(50):   1.62  |  MeanLen(50):  228.6
  [Exp 15]  Ep    9  |  Step       339  |  MeanRew(50):   1.44  |  MeanLen(50):  220.1
  [Exp 15]  Ep   10  |  Step       371  |  MeanRew(50):   1.40  

{'experiment_id': 15,
 'member': 'Noella Uwimana',
 'label': 'Very Low Gamma + Med Batch',
 'lr': 0.0001,
 'gamma': 0.9,
 'batch_size': 64,
 'eps_start': 1.0,
 'eps_end': 0.05,
 'eps_decay_steps': 100000,
 'mean_reward_last50': 16.0,
 'training_time_s': 837.9,
 'total_episodes': 1941,
 'noted_behavior': 'Very Low Gamma + Med Batch (0.90/64): Myopic; ignores long chains of bricks. Worst among gamma variants.'}

In [8]:
# ─── CELL 8: Experiment 16 — Fast Epsilon Decay (50k steps) + Med Batch ─────
# Hypothesis: Decaying epsilon too fast = commits to bad policy early.
# Expected: Agent stops exploring too soon, gets stuck in sub-optimal strategy.
run_experiment(16)


  EXPERIMENT 16: Fast Eps Decay + Med Batch
  lr=0.0001  gamma=0.99  batch=64
  epsilon: 1.0 → 0.05 over 50,000 steps
  Device: CUDA  |  Steps: 150,000
  [Exp 16]  Ep    1  |  Step        61  |  MeanRew(50):   4.00  |  MeanLen(50):  310.0
  [Exp 16]  Ep    2  |  Step       107  |  MeanRew(50):   3.00  |  MeanLen(50):  288.0
  [Exp 16]  Ep    3  |  Step       132  |  MeanRew(50):   2.00  |  MeanLen(50):  249.0
  [Exp 16]  Ep    4  |  Step       166  |  MeanRew(50):   1.75  |  MeanLen(50):  240.2
  [Exp 16]  Ep    5  |  Step       205  |  MeanRew(50):   1.80  |  MeanLen(50):  234.2
  [Exp 16]  Ep    6  |  Step       246  |  MeanRew(50):   1.83  |  MeanLen(50):  233.8
  [Exp 16]  Ep    7  |  Step       282  |  MeanRew(50):   1.71  |  MeanLen(50):  230.4
  [Exp 16]  Ep    8  |  Step       316  |  MeanRew(50):   1.62  |  MeanLen(50):  228.6
  [Exp 16]  Ep    9  |  Step       339  |  MeanRew(50):   1.44  |  MeanLen(50):  220.1
  [Exp 16]  Ep   10  |  Step       371  |  MeanRew(50):   1.40  

{'experiment_id': 16,
 'member': 'Noella Uwimana',
 'label': 'Fast Eps Decay + Med Batch',
 'lr': 0.0001,
 'gamma': 0.99,
 'batch_size': 64,
 'eps_start': 1.0,
 'eps_end': 0.05,
 'eps_decay_steps': 50000,
 'mean_reward_last50': 14.56,
 'training_time_s': 861.8,
 'total_episodes': 1681,
 'noted_behavior': 'Fast Eps Decay (50k/64): Commits to exploitation too early; gets stuck in a local strategy with no exploration fallback.'}

In [9]:
# ─── CELL 9: Experiment 17 — High Epsilon End (0.10) + Med Batch ────────────
# Hypothesis: Staying 10% random even at the end limits final performance.
# Expected: More robust but lower peak reward — never fully exploits knowledge.
run_experiment(17)


  EXPERIMENT 17: High Eps End + Med Batch
  lr=0.0001  gamma=0.99  batch=64
  epsilon: 1.0 → 0.1 over 100,000 steps
  Device: CUDA  |  Steps: 150,000
  [Exp 17]  Ep    1  |  Step        61  |  MeanRew(50):   4.00  |  MeanLen(50):  310.0
  [Exp 17]  Ep    2  |  Step       107  |  MeanRew(50):   3.00  |  MeanLen(50):  288.0
  [Exp 17]  Ep    3  |  Step       132  |  MeanRew(50):   2.00  |  MeanLen(50):  249.0
  [Exp 17]  Ep    4  |  Step       166  |  MeanRew(50):   1.75  |  MeanLen(50):  240.2
  [Exp 17]  Ep    5  |  Step       205  |  MeanRew(50):   1.80  |  MeanLen(50):  234.2
  [Exp 17]  Ep    6  |  Step       246  |  MeanRew(50):   1.83  |  MeanLen(50):  233.8
  [Exp 17]  Ep    7  |  Step       282  |  MeanRew(50):   1.71  |  MeanLen(50):  230.4
  [Exp 17]  Ep    8  |  Step       316  |  MeanRew(50):   1.62  |  MeanLen(50):  228.6
  [Exp 17]  Ep    9  |  Step       339  |  MeanRew(50):   1.44  |  MeanLen(50):  220.1
  [Exp 17]  Ep   10  |  Step       371  |  MeanRew(50):   1.40  | 

{'experiment_id': 17,
 'member': 'Noella Uwimana',
 'label': 'High Eps End + Med Batch',
 'lr': 0.0001,
 'gamma': 0.99,
 'batch_size': 64,
 'eps_start': 1.0,
 'eps_end': 0.1,
 'eps_decay_steps': 100000,
 'mean_reward_last50': 10.26,
 'training_time_s': 837.8,
 'total_episodes': 2005,
 'noted_behavior': 'High Eps End (0.10/64): Keeps exploring long after learning — reduces peak reward but improves robustness.'}

In [10]:
# ─── CELL 10: Experiment 18 — Gamma 0.97 + Large Batch (128) ────────────────
# Hypothesis: Combining slightly lower gamma with large batch is a strong combo.
# Expected: One of the better performers — smooth + moderate planning horizon.
run_experiment(18)


  EXPERIMENT 18: Slightly Low Gamma + Large Batch
  lr=0.0001  gamma=0.97  batch=128
  epsilon: 1.0 → 0.05 over 100,000 steps
  Device: CUDA  |  Steps: 150,000
  [Exp 18]  Ep    1  |  Step        61  |  MeanRew(50):   4.00  |  MeanLen(50):  310.0
  [Exp 18]  Ep    2  |  Step       107  |  MeanRew(50):   3.00  |  MeanLen(50):  288.0
  [Exp 18]  Ep    3  |  Step       132  |  MeanRew(50):   2.00  |  MeanLen(50):  249.0
  [Exp 18]  Ep    4  |  Step       166  |  MeanRew(50):   1.75  |  MeanLen(50):  240.2
  [Exp 18]  Ep    5  |  Step       205  |  MeanRew(50):   1.80  |  MeanLen(50):  234.2
  [Exp 18]  Ep    6  |  Step       246  |  MeanRew(50):   1.83  |  MeanLen(50):  233.8
  [Exp 18]  Ep    7  |  Step       282  |  MeanRew(50):   1.71  |  MeanLen(50):  230.4
  [Exp 18]  Ep    8  |  Step       316  |  MeanRew(50):   1.62  |  MeanLen(50):  228.6
  [Exp 18]  Ep    9  |  Step       339  |  MeanRew(50):   1.44  |  MeanLen(50):  220.1
  [Exp 18]  Ep   10  |  Step       371  |  MeanRew(50): 

{'experiment_id': 18,
 'member': 'Noella Uwimana',
 'label': 'Slightly Low Gamma + Large Batch',
 'lr': 0.0001,
 'gamma': 0.97,
 'batch_size': 128,
 'eps_start': 1.0,
 'eps_end': 0.05,
 'eps_decay_steps': 100000,
 'mean_reward_last50': 16.96,
 'training_time_s': 929.2,
 'total_episodes': 1887,
 'noted_behavior': 'Slightly Low Gamma + Large Batch (0.97/128): Good balance; larger batch smooths gradients, 0.97 still plans ahead.'}

In [11]:
# ─── CELL 11: Experiment 19 — Low Epsilon Start (0.5) ───────────────────────
# Hypothesis: Starting with 50% exploitation skips valuable early exploration.
# Expected: Agent misses state space coverage, underperforms vs full exploration.
run_experiment(19)


  EXPERIMENT 19: Low Eps Start
  lr=0.0001  gamma=0.99  batch=32
  epsilon: 0.5 → 0.05 over 100,000 steps
  Device: CUDA  |  Steps: 150,000
  [Exp 19]  Ep    1  |  Step        61  |  MeanRew(50):   4.00  |  MeanLen(50):  310.0
  [Exp 19]  Ep    2  |  Step       107  |  MeanRew(50):   3.00  |  MeanLen(50):  288.0
  [Exp 19]  Ep    3  |  Step       132  |  MeanRew(50):   2.00  |  MeanLen(50):  249.0
  [Exp 19]  Ep    4  |  Step       166  |  MeanRew(50):   1.75  |  MeanLen(50):  240.2
  [Exp 19]  Ep    5  |  Step       205  |  MeanRew(50):   1.80  |  MeanLen(50):  234.2
  [Exp 19]  Ep    6  |  Step       246  |  MeanRew(50):   1.83  |  MeanLen(50):  233.8
  [Exp 19]  Ep    7  |  Step       282  |  MeanRew(50):   1.71  |  MeanLen(50):  230.4
  [Exp 19]  Ep    8  |  Step       316  |  MeanRew(50):   1.62  |  MeanLen(50):  228.6
  [Exp 19]  Ep    9  |  Step       339  |  MeanRew(50):   1.44  |  MeanLen(50):  220.1
  [Exp 19]  Ep   10  |  Step       371  |  MeanRew(50):   1.40  |  MeanLen(5

{'experiment_id': 19,
 'member': 'Noella Uwimana',
 'label': 'Low Eps Start',
 'lr': 0.0001,
 'gamma': 0.99,
 'batch_size': 32,
 'eps_start': 0.5,
 'eps_end': 0.05,
 'eps_decay_steps': 100000,
 'mean_reward_last50': 14.2,
 'training_time_s': 839.1,
 'total_episodes': 1701,
 'noted_behavior': 'Low Eps Start (0.5/32): Agent exploits too early with a small batch — misses important early exploration.'}

In [12]:
# ─── CELL 12: Experiment 20 — BEST CONFIG ───────────────────────────────────
# gamma=0.97, batch=64, eps_end=0.02, eps_decay=120k
# Hypothesis: Best balance of exploration, exploitation, and sample stability.
# Expected: Highest mean reward of all 10 experiments.
run_experiment(20)


  EXPERIMENT 20: Best — Low Gamma + Med Batch + Low Eps End
  lr=0.0001  gamma=0.97  batch=64
  epsilon: 1.0 → 0.02 over 120,000 steps
  Device: CUDA  |  Steps: 150,000
  [Exp 20]  Ep    1  |  Step        61  |  MeanRew(50):   4.00  |  MeanLen(50):  310.0
  [Exp 20]  Ep    2  |  Step       107  |  MeanRew(50):   3.00  |  MeanLen(50):  288.0
  [Exp 20]  Ep    3  |  Step       132  |  MeanRew(50):   2.00  |  MeanLen(50):  249.0
  [Exp 20]  Ep    4  |  Step       166  |  MeanRew(50):   1.75  |  MeanLen(50):  240.2
  [Exp 20]  Ep    5  |  Step       205  |  MeanRew(50):   1.80  |  MeanLen(50):  234.2
  [Exp 20]  Ep    6  |  Step       246  |  MeanRew(50):   1.83  |  MeanLen(50):  233.8
  [Exp 20]  Ep    7  |  Step       282  |  MeanRew(50):   1.71  |  MeanLen(50):  230.4
  [Exp 20]  Ep    8  |  Step       316  |  MeanRew(50):   1.62  |  MeanLen(50):  228.6
  [Exp 20]  Ep    9  |  Step       339  |  MeanRew(50):   1.44  |  MeanLen(50):  220.1
  [Exp 20]  Ep   10  |  Step       371  |  Mean

{'experiment_id': 20,
 'member': 'Noella Uwimana',
 'label': 'Best — Low Gamma + Med Batch + Low Eps End',
 'lr': 0.0001,
 'gamma': 0.97,
 'batch_size': 64,
 'eps_start': 1.0,
 'eps_end': 0.02,
 'eps_decay_steps': 120000,
 'mean_reward_last50': 15.36,
 'training_time_s': 842.1,
 'total_episodes': 2041,
 'noted_behavior': 'BEST — Low Gamma + Med Batch + Low Eps End (0.97/64/0.02): Ideal mix: enough exploration early, tight exploitation late.'}

In [13]:
# ─── CELL 13: Leaderboard + Summary (run after ALL experiments) ─────────────

if not ALL_RESULTS:
    print('❌ No results yet — run the experiment cells first!')
else:
    # Print table
    print(f'\n  HYPERPARAMETER TABLE — {MEMBER_NAME}')
    print('  ' + '─'*90)
    print(f'  {"#":<4} {"lr":<8} {"gamma":<7} {"batch":<7} {"eps_s":<6} {"eps_e":<6} {"eps_decay":<10} {"reward":<8} label')
    print('  ' + '─'*90)
    for r in ALL_RESULTS:
        print(f'  {r["experiment_id"]:<4} '
              f'{str(r["lr"]):<8} '
              f'{r["gamma"]:<7} '
              f'{r["batch_size"]:<7} '
              f'{r["eps_start"]:<6} '
              f'{r["eps_end"]:<6} '
              f'{r["eps_decay_steps"]:<10} '
              f'{r["mean_reward_last50"]:<8.2f} '
              f'{r["label"]}')
    print('  ' + '─'*90)

    # Leaderboard
    ranked = sorted(ALL_RESULTS, key=lambda r: r['mean_reward_last50'], reverse=True)
    print(f'\n  🏆 LEADERBOARD')
    for rank, r in enumerate(ranked, 1):
        medal = ['🥇','🥈','🥉'][rank-1] if rank <= 3 else f'#{rank}'
        print(f'  {medal}  Exp {r["experiment_id"]:>2}  '
              f'Reward: {r["mean_reward_last50"]:>6.2f}  |  '
              f'{r["label"]}')

    # Save summary CSV
    summary_path = os.path.join(LOG_DIR, 'experiments_summary_noella.csv')
    with open(summary_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=ALL_RESULTS[0].keys())
        writer.writeheader()
        writer.writerows(ALL_RESULTS)
    print(f'\n  💾 Summary CSV saved → {summary_path}')

    # Save best model as dqn_model.zip
    best     = ranked[0]
    best_src = os.path.join(LOG_DIR, f'dqn_exp_{best["experiment_id"]:02d}.zip')
    best_model = DQN.load(best_src, device=device)
    best_model.save(MODEL_SAVE_PATH)
    print(f'  🏆 Best model (Exp {best["experiment_id"]}) saved → {MODEL_SAVE_PATH}.zip')
    print(f'     {best["label"]}  |  Mean Reward: {best["mean_reward_last50"]:.2f}')


  HYPERPARAMETER TABLE — Noella Uwimana
  ──────────────────────────────────────────────────────────────────────────────────────────
  #    lr       gamma   batch   eps_s  eps_e  eps_decay  reward   label
  ──────────────────────────────────────────────────────────────────────────────────────────
  11   0.0001   0.99    16      1.0    0.05   100000     9.34     Small Batch (16)
  12   0.0001   0.99    128     1.0    0.05   100000     17.18    Large Batch (128)
  13   0.0001   0.99    256     1.0    0.05   100000     14.52    XL Batch (256)
  14   0.0001   0.95    64      1.0    0.05   100000     18.04    Low Gamma + Med Batch
  15   0.0001   0.9     64      1.0    0.05   100000     16.00    Very Low Gamma + Med Batch
  16   0.0001   0.99    64      1.0    0.05   50000      14.56    Fast Eps Decay + Med Batch
  17   0.0001   0.99    64      1.0    0.1    100000     10.26    High Eps End + Med Batch
  18   0.0001   0.97    128     1.0    0.05   100000     16.96    Slightly Low Gamma + L

In [14]:
# ─── CELL 14: Download your results from Kaggle ─────────────────────────────
# This zips everything so you can download it in one click from the Output tab.

import shutil

zip_path = '/kaggle/working/noella_results'
shutil.make_archive(zip_path, 'zip', LOG_DIR)
print(f'✅ All results zipped → {zip_path}.zip')
print()
print('To download:')
print('  1. Click the folder icon on the left sidebar in Kaggle')
print('  2. Go to /kaggle/working/')
print('  3. Click the three dots next to noella_results.zip → Download')
print()
print('Files inside the zip:')
for f in sorted(os.listdir(LOG_DIR)):
    size = os.path.getsize(os.path.join(LOG_DIR, f))
    print(f'  {f:<45} {size/1024:.1f} KB')

✅ All results zipped → /kaggle/working/noella_results.zip

To download:
  1. Click the folder icon on the left sidebar in Kaggle
  2. Go to /kaggle/working/
  3. Click the three dots next to noella_results.zip → Download

Files inside the zip:
  dqn_exp_11.zip                                26602.3 KB
  dqn_exp_12.zip                                26602.4 KB
  dqn_exp_13.zip                                26602.4 KB
  dqn_exp_14.zip                                26602.4 KB
  dqn_exp_15.zip                                26602.4 KB
  dqn_exp_16.zip                                26602.4 KB
  dqn_exp_17.zip                                26602.4 KB
  dqn_exp_18.zip                                26602.4 KB
  dqn_exp_19.zip                                26602.4 KB
  dqn_exp_20.zip                                26602.3 KB
  dqn_model.zip                                 26602.5 KB
  exp_11_best                                   4.0 KB
  exp_11_eval                                   4.0 